# Ultimate TS Forecasting v2 — LGB+XGB Ensemble

## Fixes applied:
- **Target leakage fixed**: concat train+test before feature engineering (like v6)
- **Vectorized FE**: no slow lambda transforms, proper shift(1) before rolling
- **Removed CatBoost**: saves ~2h runtime, was problematic for H=10,25
- **Removed walk-forward CV**: simple early stopping (proven in v6)
- **Removed rolling slope**: too slow (np.polyfit per window)
- **Removed one-hot sub_category**: trees handle categoricals natively
- **Fixed meta-stacking**: removed Ridge (overfitting on val), using optimal blend + calibration
- **Fixed XGB**: own early stopping instead of using LGB's best_iter
- **Runtime target**: ~5h (under 6h limit)

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import LinearRegression
import gc, os, time, warnings

warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None
T0 = time.time()

print(f'LightGBM : {lgb.__version__}')
print(f'XGBoost  : {xgb.__version__}')
print('Imports  : OK')

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
TRAIN_PATH = '/kaggle/input/competitions/ts-forecasting/train.parquet'
TEST_PATH  = '/kaggle/input/competitions/ts-forecasting/test.parquet'
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = '/kaggle/input/ts-forecasting/train.parquet'
    TEST_PATH  = '/kaggle/input/ts-forecasting/test.parquet'

HORIZONS      = [1, 3, 10, 25]
VAL_THRESHOLD = 3500
TARGET = 'y_target'
WEIGHT = 'weight'
GROUP_COLS = ['code', 'sub_code', 'sub_category', 'horizon']

# Top features for lag/rolling (expanded from v6's 5)
KEY_FEATURES   = ['feature_al', 'feature_am', 'feature_cg', 'feature_by', 'feature_s']
EXTRA_FEATURES = ['feature_t', 'feature_j', 'feature_bz', 'feature_cd', 'feature_a']
LAG_STEPS      = [1, 3, 5, 10, 25]
ROLLING_WINDOWS = [5, 10, 20]

# Seeds
LGB_SEEDS = [42, 2024, 12345, 99, 420, 777, 1337, 2025, 7, 11, 314, 617]  # 12
XGB_SEEDS = [42, 2024, 12345]  # 3

# Per-horizon LightGBM params
HORIZON_LGB = {
    1:  {'num_leaves': 63,  'min_child_samples': 300, 'lambda_l2': 15.0},
    3:  {'num_leaves': 70,  'min_child_samples': 250, 'lambda_l2': 12.0},
    10: {'num_leaves': 90,  'min_child_samples': 200, 'lambda_l2': 10.0},
    25: {'num_leaves': 100, 'min_child_samples': 150, 'lambda_l2': 8.0},
}
BASE_LGB = {
    'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
    'learning_rate': 0.015, 'feature_fraction': 0.65, 'bagging_fraction': 0.75,
    'bagging_freq': 5, 'lambda_l1': 1.0, 'extra_trees': True,
    'path_smooth': 1.0, 'verbosity': -1, 'n_jobs': -1,
}

# Per-horizon XGBoost params
HORIZON_XGB = {
    1:  {'max_depth': 6, 'min_child_weight': 300, 'reg_lambda': 15.0},
    3:  {'max_depth': 6, 'min_child_weight': 250, 'reg_lambda': 12.0},
    10: {'max_depth': 7, 'min_child_weight': 200, 'reg_lambda': 10.0},
    25: {'max_depth': 7, 'min_child_weight': 150, 'reg_lambda': 8.0},
}
BASE_XGB = {
    'objective': 'reg:squarederror', 'eval_metric': 'rmse',
    'learning_rate': 0.015, 'subsample': 0.75, 'colsample_bytree': 0.65,
    'reg_alpha': 1.0, 'tree_method': 'hist', 'verbosity': 0,
}

print(f'LGB seeds: {len(LGB_SEEDS)} | XGB seeds: {len(XGB_SEEDS)}')
print(f'Horizons: {HORIZONS}')

In [ ]:
# ============================================================
# METRIC + BLEND
# ============================================================
def comp_score(y_true, y_pred, w):
    y_true, y_pred, w = [np.asarray(x, dtype=np.float64) for x in [y_true, y_pred, w]]
    denom = np.sum(w * y_true ** 2)
    if denom < 1e-15:
        return 0.0
    ratio = np.sum(w * (y_true - y_pred) ** 2) / denom
    return float(np.sqrt(1.0 - min(max(ratio, 0.0), 1.0)))


def find_blend(p1, p2, y, w, n=50):
    best_s, best_a = -1, 0.5
    for i in range(n + 1):
        a = i / n
        s = comp_score(y, a * p1 + (1 - a) * p2, w)
        if s > best_s:
            best_s, best_a = s, a
    return best_a, best_s


# Target transform for H=25 (reduce outlier impact)
def transform_y(y, h):
    if h == 25:
        return np.sign(y) * np.log1p(np.abs(y))
    return y

def inv_transform_y(y, h):
    if h == 25:
        return np.sign(y) * np.expm1(np.abs(y))
    return y

print('Metric, blend, target transform OK')

In [ ]:
# ============================================================
# ENCODING STATS (train-only, no leakage)
# ============================================================
def compute_target_stats(df):
    return {
        'code': df.groupby('code')[TARGET].mean().to_dict(),
        'sub_category': df.groupby('sub_category')[TARGET].mean().to_dict(),
        'sub_code': df.groupby('sub_code')[TARGET].mean().to_dict(),
        'global_mean': float(df[TARGET].mean()),
        'sub_code_q10': df.groupby('sub_code')[TARGET].quantile(0.10).to_dict(),
        'sub_code_q90': df.groupby('sub_code')[TARGET].quantile(0.90).to_dict(),
        'sub_code_std': df.groupby('sub_code')[TARGET].std().fillna(1.0).to_dict(),
        'sub_cat_std': df.groupby('sub_category')[TARGET].std().fillna(1.0).to_dict(),
    }

def compute_freq_encoding(df):
    freq = {}
    for c in ['code', 'sub_code', 'sub_category']:
        if c in df.columns:
            freq[c] = df[c].value_counts(normalize=True).to_dict()
    return freq

print('Encoding stats functions OK')

In [ ]:
# ============================================================
# FEATURE ENGINEERING — vectorized, concat-safe, no leakage
# ============================================================
def build_features(data, stats, freq_stats):
    df = data.copy()
    gm = stats['global_mean']
    raw_cols = [c for c in df.columns if c.startswith('feature_')]

    # ── 1. Row-wise meta-features (numpy for speed) ──
    X = df[raw_cols].fillna(0.0).values
    df['feat_mean']     = np.mean(X, axis=1).astype(np.float32)
    df['feat_std']      = np.std(X, axis=1).astype(np.float32)
    df['feat_range']    = (np.max(X, axis=1) - np.min(X, axis=1)).astype(np.float32)
    df['feat_pos_frac'] = (X > 0).mean(axis=1).astype(np.float32)
    df['feat_l2']       = np.sqrt((X ** 2).sum(axis=1)).astype(np.float32)
    del X

    # ── 2. Feature Interactions (expanded) ──
    E = lambda f: f in df.columns
    if E('feature_al') and E('feature_am'):
        df['d_al_am'] = df['feature_al'] - df['feature_am']
        df['r_al_am'] = df['feature_al'] / (df['feature_am'].abs() + 1e-7)
        df['p_al_am'] = df['feature_al'] * df['feature_am']
    if E('feature_cg') and E('feature_by'):
        df['d_cg_by'] = df['feature_cg'] - df['feature_by']
        df['r_cg_by'] = df['feature_cg'] / (df['feature_by'].abs() + 1e-7)
    if E('feature_s') and E('feature_t'):
        df['d_s_t'] = df['feature_s'] - df['feature_t']
    if E('feature_s'):
        for f in ['feature_al', 'feature_am', 'feature_cg']:
            if E(f):
                df[f's_{f.split("_")[1]}_prod'] = df['feature_s'] * df[f]
    if E('feature_am') and E('feature_bz'):
        df['p_am_bz'] = df['feature_am'] * df['feature_bz']
    if E('feature_al') and E('feature_cg'):
        df['al_x_cg'] = df['feature_al'] * df['feature_cg']
    if E('feature_a') and E('feature_b'):
        df['d_a_b'] = df['feature_a'] - df['feature_b']
    if E('feature_c') and E('feature_d'):
        df['d_c_d'] = df['feature_c'] - df['feature_d']
    if E('feature_e') and E('feature_f'):
        df['d_e_f'] = df['feature_e'] - df['feature_f']

    # ── 3. Target Encoding (from train stats) ──
    for c in ['code', 'sub_category', 'sub_code']:
        if c in stats:
            df[c + '_enc'] = df[c].map(stats[c]).fillna(gm).astype(np.float32)
    df['sc_q10']    = df['sub_code'].map(stats.get('sub_code_q10', {})).fillna(gm).astype(np.float32)
    df['sc_q90']    = df['sub_code'].map(stats.get('sub_code_q90', {})).fillna(gm).astype(np.float32)
    df['sc_qrange'] = (df['sc_q90'] - df['sc_q10']).astype(np.float32)
    df['sc_std']    = df['sub_code'].map(stats.get('sub_code_std', {})).fillna(1.0).astype(np.float32)
    df['scat_std']  = df['sub_category'].map(stats.get('sub_cat_std', {})).fillna(1.0).astype(np.float32)
    df['code_snr']  = (df['sub_code_enc'].abs() / (df['sc_std'] + 1e-7)).astype(np.float32)

    # ── 4. Frequency Encoding ──
    for c in ['code', 'sub_code', 'sub_category']:
        if c in freq_stats:
            df[c + '_freq'] = df[c].map(freq_stats[c]).fillna(0).astype(np.float32)
    df['sc_log_freq'] = np.log1p(df.get('sub_code_freq', pd.Series(0, index=df.index))).astype(np.float32)

    # ── 5. Cross-sectional normalization ──
    cs_cols = [c for c in ['feature_al', 'feature_am', 'feature_cg', 'feature_by',
                           'd_al_am', 'd_cg_by', 'feat_mean'] if c in df.columns]
    for col in cs_cols:
        g = df.groupby('ts_index')[col]
        df[col + '_cs'] = ((df[col] - g.transform('mean')) / (g.transform('std') + 1e-7)).astype(np.float32)
        df[col + '_rank'] = df.groupby('ts_index')[col].rank(pct=True).astype(np.float32)
    for f in ['feature_al', 'feature_am', 'd_al_am']:
        if f in df.columns:
            g2 = df.groupby(['ts_index', 'sub_category'])[f]
            df[f + '_gcs'] = ((df[f] - g2.transform('mean')) / (g2.transform('std') + 1e-7)).astype(np.float32)

    # ── 6. Time features ──
    ts = df['ts_index']
    for p in [2, 3, 5, 7, 12, 24, 30]:
        df[f'ts_mod_{p}'] = (ts % p).astype(np.int8)
    df['t_sin']  = np.sin(2 * np.pi * ts / 100).astype(np.float32)
    df['t_cos']  = np.cos(2 * np.pi * ts / 100).astype(np.float32)
    df['t_sin2'] = np.sin(2 * np.pi * ts / 52).astype(np.float32)
    df['t_cos2'] = np.cos(2 * np.pi * ts / 52).astype(np.float32)
    df['ts_norm'] = (ts / 4000.0).astype(np.float32)
    df['horizon_log'] = np.log1p(df['horizon']).astype(np.float32)

    # ── 7. Lifecycle ──
    df = df.sort_values(GROUP_COLS + ['ts_index'])
    df['obs_idx'] = df.groupby(GROUP_COLS).cumcount().astype(np.int32)
    first_t = df.groupby(GROUP_COLS)['ts_index'].transform('min')
    df['time_since_start'] = (df['ts_index'] - first_t).astype(np.int32)
    del first_t

    # ── 8. Lags, Rolling, EWM, Diff — VECTORIZED ──
    key_present   = [f for f in KEY_FEATURES if f in df.columns]
    extra_present = [f for f in EXTRA_FEATURES if f in df.columns]
    inter_feats   = [f for f in ['d_al_am', 'd_cg_by', 'd_s_t'] if f in df.columns]
    grouped = df.groupby(GROUP_COLS, sort=False)
    new = {}

    # Full treatment for key features
    for col in key_present:
        for lag in LAG_STEPS:
            new[f'{col}_lag{lag}'] = grouped[col].shift(lag).astype(np.float32)
        for w in ROLLING_WINDOWS:
            shifted = grouped[col].shift(1)
            new[f'{col}_rmean_{w}'] = shifted.rolling(w, min_periods=1).mean().astype(np.float32)
            new[f'{col}_rstd_{w}']  = shifted.rolling(w, min_periods=1).std().astype(np.float32)
        new[f'{col}_ewm10'] = grouped[col].shift(1).ewm(span=10, min_periods=1).mean().values.astype(np.float32)
        new[f'{col}_diff1'] = grouped[col].diff(1).astype(np.float32)
        new[f'{col}_rank']  = df.groupby('ts_index')[col].rank(pct=True).astype(np.float32)

    # Lighter treatment for extra features
    for col in extra_present:
        for lag in [1, 3, 5]:
            new[f'{col}_lag{lag}'] = grouped[col].shift(lag).astype(np.float32)
        new[f'{col}_diff1'] = grouped[col].diff(1).astype(np.float32)
        shifted = grouped[col].shift(1)
        new[f'{col}_rmean_5'] = shifted.rolling(5, min_periods=1).mean().astype(np.float32)
        new[f'{col}_ewm10']   = grouped[col].shift(1).ewm(span=10, min_periods=1).mean().values.astype(np.float32)

    # Interaction features: basic lags
    for col in inter_feats:
        for lag in [1, 3]:
            new[f'{col}_lag{lag}'] = grouped[col].shift(lag).astype(np.float32)
        new[f'{col}_diff1'] = grouped[col].diff(1).astype(np.float32)

    if new:
        df = pd.concat([df, pd.DataFrame(new, index=df.index)], axis=1)

    # ── 9. Momentum ──
    for col in key_present[:4]:
        l1, l5, rm5 = f'{col}_lag1', f'{col}_lag5', f'{col}_rmean_5'
        if l1 in df.columns and l5 in df.columns:
            df[f'{col}_mom15'] = (df[l1] - df[l5]).astype(np.float32)
        if l1 in df.columns and rm5 in df.columns:
            df[f'{col}_dev5'] = (df[l1] - df[rm5]).astype(np.float32)

    # ── 10. Fill NaN/Inf ──
    preserved_y = df[TARGET].copy() if TARGET in df.columns else None
    preserved_w = df[WEIGHT].copy() if WEIGHT in df.columns else None
    df = df.fillna(0.0).replace([np.inf, -np.inf], 0.0)
    if preserved_y is not None: df[TARGET] = preserved_y
    if preserved_w is not None: df[WEIGHT] = preserved_w

    gc.collect()
    return df


def get_feat_cols(df):
    exclude = {'id', 'code', 'sub_code', 'sub_category', 'horizon', 'ts_index', 'weight', 'y_target'}
    return [c for c in df.columns if c not in exclude]

print('Feature engineering functions OK')

## Train + Predict

In [ ]:
# ============================================================
# SOLVE ONE HORIZON
# ============================================================
def solve_horizon(horizon):
    t0 = time.time()
    print(f'\n{"="*60}')
    print(f'HORIZON {horizon}')
    print(f'{"="*60}')

    # ── Load data ──
    tr = pd.read_parquet(TRAIN_PATH).query('horizon == @horizon').reset_index(drop=True)
    te = pd.read_parquet(TEST_PATH).query('horizon == @horizon').reset_index(drop=True)
    print(f'Data: train={len(tr):,}, test={len(te):,}')

    # ── Stats from train only (no leakage) ──
    target_stats = compute_target_stats(tr)

    # ── FIX: Concat train+test for continuous lags ──
    te_tmp = te.copy()
    te_tmp[TARGET] = np.nan
    combined = pd.concat([tr, te_tmp], ignore_index=True)
    combined = combined.sort_values(GROUP_COLS + ['ts_index']).reset_index(drop=True)
    freq_stats = compute_freq_encoding(combined)

    print('Building features...')
    combined_feat = build_features(combined, target_stats, freq_stats)

    is_train = combined_feat[TARGET].notna()
    all_feat = combined_feat[is_train].reset_index(drop=True)
    te_feat  = combined_feat[~is_train].reset_index(drop=True)
    print(f'Split: train_feat={len(all_feat):,}, test_feat={len(te_feat):,}')
    del combined, combined_feat, te_tmp
    gc.collect()

    feats = get_feat_cols(all_feat)
    for c in feats:
        if c not in te_feat.columns:
            te_feat[c] = 0.0
    print(f'Features: {len(feats)}')

    # ── Train/Val split ──
    tr_mask = all_feat['ts_index'] <= VAL_THRESHOLD
    va_mask = all_feat['ts_index'] > VAL_THRESHOLD
    X_tr = all_feat.loc[tr_mask, feats]
    y_tr_raw = all_feat.loc[tr_mask, TARGET].values
    w_tr = all_feat.loc[tr_mask, WEIGHT].values
    X_va = all_feat.loc[va_mask, feats]
    y_va_raw = all_feat.loc[va_mask, TARGET].values
    w_va = all_feat.loc[va_mask, WEIGHT].values
    print(f'Train: {len(X_tr):,}, Val: {len(X_va):,}')

    y_tr = transform_y(y_tr_raw, horizon)
    y_va = transform_y(y_va_raw, horizon)

    # ═══════════════════════════════════════════
    # PROBE LGB — early stopping
    # ═══════════════════════════════════════════
    p_lgb = {**BASE_LGB, **HORIZON_LGB[horizon], 'n_estimators': 5000, 'random_state': 42}
    probe_lgb = lgb.LGBMRegressor(**p_lgb)
    probe_lgb.fit(X_tr, y_tr, sample_weight=w_tr,
                  eval_set=[(X_va, y_va)], eval_sample_weight=[w_va],
                  callbacks=[lgb.early_stopping(200, verbose=False)])
    best_iter_lgb = max(probe_lgb.best_iteration_ or 3000, 50)
    val_pred_lgb = inv_transform_y(probe_lgb.predict(X_va), horizon)
    lgb_score = comp_score(y_va_raw, val_pred_lgb, w_va)
    print(f'[LGB] Val={lgb_score:.6f}, best_iter={best_iter_lgb}')
    del probe_lgb; gc.collect()

    # ═══════════════════════════════════════════
    # PROBE XGB — own early stopping
    # ═══════════════════════════════════════════
    p_xgb = {**BASE_XGB, **HORIZON_XGB[horizon], 'n_estimators': 5000,
             'random_state': 42, 'early_stopping_rounds': 200}
    probe_xgb = xgb.XGBRegressor(**p_xgb)
    probe_xgb.fit(X_tr, y_tr, sample_weight=w_tr,
                  eval_set=[(X_va, y_va)], sample_weight_eval_set=[w_va],
                  verbose=False)
    best_iter_xgb = max(probe_xgb.best_iteration_ or 3000, 50)
    val_pred_xgb = inv_transform_y(probe_xgb.predict(X_va), horizon)
    xgb_score = comp_score(y_va_raw, val_pred_xgb, w_va)
    print(f'[XGB] Val={xgb_score:.6f}, best_iter={best_iter_xgb}')
    del probe_xgb; gc.collect()

    # ═══════════════════════════════════════════
    # BLEND — optimal weights on validation
    # ═══════════════════════════════════════════
    blend_a, blend_score = find_blend(val_pred_lgb, val_pred_xgb, y_va_raw, w_va)
    print(f'[Blend] LGB={blend_a:.2f} XGB={1-blend_a:.2f}  score={blend_score:.6f}')
    val_pred = blend_a * val_pred_lgb + (1 - blend_a) * val_pred_xgb

    # ═══════════════════════════════════════════
    # CALIBRATION — linear a*pred + b
    # ═══════════════════════════════════════════
    calibrator = LinearRegression()
    calibrator.fit(val_pred.reshape(-1, 1), y_va_raw, sample_weight=w_va)
    cal_val = calibrator.predict(val_pred.reshape(-1, 1)).ravel()
    cal_score = comp_score(y_va_raw, cal_val, w_va)
    use_cal = cal_score >= blend_score
    if use_cal:
        print(f'[Calib] a={calibrator.coef_[0]:.4f} b={calibrator.intercept_:.6f}  score={cal_score:.6f} (applied)')
        final_val = cal_val
        final_score = cal_score
    else:
        print(f'[Calib] score={cal_score:.6f} < blend={blend_score:.6f} (skipped)')
        final_val = val_pred
        final_score = blend_score

    # ═══════════════════════════════════════════
    # RETRAIN on ALL data
    # ═══════════════════════════════════════════
    X_all = all_feat[feats]
    y_all_raw = all_feat[TARGET].values
    y_all = transform_y(y_all_raw, horizon)
    w_all = all_feat[WEIGHT].values

    # LGB retrain
    lgb_test = np.zeros(len(te_feat), dtype=np.float64)
    for i, seed in enumerate(LGB_SEEDS):
        if i == 0 or (i + 1) % 4 == 0:
            print(f'  LGB seed {i+1}/{len(LGB_SEEDS)}...')
        rp = {**BASE_LGB, **HORIZON_LGB[horizon], 'n_estimators': best_iter_lgb, 'random_state': seed}
        mdl = lgb.LGBMRegressor(**rp)
        mdl.fit(X_all, y_all, sample_weight=w_all)
        lgb_test += inv_transform_y(mdl.predict(te_feat[feats]), horizon) / len(LGB_SEEDS)
        del mdl; gc.collect()

    # XGB retrain
    xgb_test = np.zeros(len(te_feat), dtype=np.float64)
    for i, seed in enumerate(XGB_SEEDS):
        print(f'  XGB seed {i+1}/{len(XGB_SEEDS)}...')
        rp = {**BASE_XGB, **HORIZON_XGB[horizon], 'n_estimators': best_iter_xgb, 'random_state': seed}
        mdl = xgb.XGBRegressor(**rp)
        mdl.fit(X_all, y_all, sample_weight=w_all)
        xgb_test += inv_transform_y(mdl.predict(te_feat[feats]), horizon) / len(XGB_SEEDS)
        del mdl; gc.collect()

    # Blend + calibrate test
    test_pred = blend_a * lgb_test + (1 - blend_a) * xgb_test
    if use_cal:
        test_pred = calibrator.predict(test_pred.reshape(-1, 1)).ravel()

    # Clip
    clip_lo = np.percentile(y_all_raw, 0.5)
    clip_hi = np.percentile(y_all_raw, 99.5)
    test_pred = np.clip(test_pred, clip_lo, clip_hi)
    print(f'Clip: [{clip_lo:.2f}, {clip_hi:.2f}]')

    elapsed = (time.time() - t0) / 60
    print(f'Horizon {horizon} done in {elapsed:.1f} min | Final CV={final_score:.6f}')

    del all_feat, te_feat, X_tr, X_va, X_all
    gc.collect()

    return {
        'horizon': horizon, 'ids': te['id'].values,
        'pred': test_pred, 'val_score': final_score,
        'val_y': y_va_raw, 'val_w': w_va, 'val_pred': final_val,
        'lgb_score': lgb_score, 'xgb_score': xgb_score,
        'blend_w': (blend_a, 1-blend_a),
    }

print('solve_horizon() OK')

In [ ]:
# ============================================================
# RUN ALL HORIZONS
# ============================================================
print('\n' + '='*60)
print('ULTIMATE TS v2 — LGB+XGB Ensemble')
print('='*60)

results = []
for h in HORIZONS:
    results.append(solve_horizon(h))

print('\nAll horizons complete')

In [ ]:
# ============================================================
# FINAL SUBMISSION
# ============================================================
sub_parts = [pd.DataFrame({'id': r['ids'], 'prediction': r['pred']}) for r in results]
sub = pd.concat(sub_parts, ignore_index=True)

all_y = np.concatenate([r['val_y'] for r in results])
all_w = np.concatenate([r['val_w'] for r in results])
all_p = np.concatenate([r['val_pred'] for r in results])
agg_score = comp_score(all_y, all_p, all_w)

sub.to_csv('submission.csv', index=False)

total = time.time() - T0
print(f'\n{"="*60}')
print(f'RESULTS')
print(f'{"H":>4} | {"LGB":>10} | {"XGB":>10} | {"Blend":>10} | {"Weights":>14}')
print('-' * 56)
for r in sorted(results, key=lambda x: x['horizon']):
    wl, wx = r['blend_w']
    print(f"  {r['horizon']:>2} | {r['lgb_score']:.6f} | {r['xgb_score']:.6f} | {r['val_score']:.6f} | L={wl:.0%} X={wx:.0%}")
print('-' * 56)
print(f'  Aggregate: {agg_score:.6f}')
print(f'{"="*60}')
print(f'Runtime: {total/60:.1f} min')
print(f'Saved submission.csv ({len(sub):,} rows)')
print(sub.head())